In [27]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, root_mean_squared_error, r2_score, precision_score, recall_score, f1_score, accuracy_score
import pickle #me permite guardar objetos de python
from sklearn.tree import DecisionTreeClassifier
import matplotlib.pyplot as plt
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

## Enfoque del Sistema de Recomendación

El sistema implementado corresponde a un enfoque de filtrado basado en contenido.

Cada individuo es representado como un vector de características socioeconómicas (edad, educación, ocupación, horas trabajadas, estado civil).

El modelo de clasificación estima la probabilidad de superar el umbral de 50K USD anuales a partir de estas características.

Posteriormente, se simulan modificaciones en variables relevantes del perfil y se evalúa cómo cambia la probabilidad estimada.

Las recomendaciones se generan en función del impacto que dichas modificaciones tienen sobre la predicción del modelo.

In [28]:
df =  pd.read_csv('/workspaces/Antonio27M-machine-learning/data/processed/eda_adult_income.csv')
df.head()

,age,fnlwgt,education,education.num,marital.status,occupation,relationship,hours.per.week,income
0,82,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,18,<=50K
1,54,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,40,<=50K
2,41,264663,Some-college,10,Separated,Prof-specialty,Own-child,40,<=50K
3,34,216864,HS-grad,9,Divorced,Other-service,Unmarried,45,<=50K
4,38,150601,10th,6,Separated,Adm-clerical,Unmarried,40,<=50K


In [29]:
df.shape, df.columns

((30685, 9),
 Index(['age', 'fnlwgt', 'education', 'education.num', 'marital.status',
        'occupation', 'relationship', 'hours.per.week', 'income'],
       dtype='object'))

In [30]:
X = df.drop("income", axis=1)
y = df["income"].apply(lambda x: 1 if x == ">50K" else 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=18, stratify=y)

In [31]:
categorical_cols = X.select_dtypes(include="object").columns
numeric_cols = X.select_dtypes(exclude="object").columns

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols)])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))])

model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [32]:
model.score(X_test, y_test)

0.8114714029656184

In [33]:
usuario = X_test.iloc[[0]].copy()
prob_actual = model.predict_proba(usuario)
prob_actual

array([[0.35, 0.65]])

In [34]:
usuario_bachelors = usuario.copy()
usuario_bachelors["education"] = "Bachelors"

prob_bachelors = model.predict_proba(usuario_bachelors)

prob_bachelors

array([[0.34, 0.66]])

In [35]:
usuario_mas_horas = usuario.copy()
usuario_mas_horas["hours.per.week"] = 50

prob_mas_horas = model.predict_proba(usuario_mas_horas)

prob_mas_horas

array([[0.35, 0.65]])

In [ ]:
resultados = pd.DataFrame({"Escenario": ["Original", "Bachelors", "Más horas"],"Probabilidad >50K": 
        [prob_actual[0][1],
        prob_bachelors[0][1],
        prob_mas_horas[0][1]]})
resultados

,Escenario,Probabilidad >50K
0,Original,0.65
1,Bachelors,0.66
2,Más horas,0.65


El usuario seleccionado presenta inicialmente una probabilidad del 65% de superar el umbral de 50K USD anuales.

La simulación de un aumento en el nivel educativo a "Bachelors" incrementa levemente esta probabilidad a 66%.

El aumento en las horas trabajadas no genera un cambio significativo en la predicción.

Esto sugiere que el perfil original del usuario ya posee características asociadas a ingresos altos y que los cambios simulados no representan una modificación estructural significativa.